# 04 — Retention Analysis

This notebook focuses specifically on retention/renewal patterns and client risk analysis.

## Objectives
- Compare engagement between renewed and non-renewed clients
- Identify at-risk clients based on activity patterns
- Analyze engagement trends near renewal dates
- Segment analysis by retention outcomes

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

plt.style.use('seaborn-v0_8-whitegrid')

PROCESSED_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "data", "processed")

features_df = pd.read_csv(os.path.join(PROCESSED_DIR, "customer_features_vB.csv"))
print(f"Loaded {len(features_df)} customer records.")

## 2. Renewed vs Non-Renewed: Engagement Comparison

In [ ]:
comparison_cols = [
    'engagement_score', 'total_interactions', 'avg_duration',
    'total_time_spent', 'total_reports', 'events_attended',
    'total_logins', 'total_sessions', 'total_features',
    'total_active_days',
]

comparison = features_df.groupby('churn')[comparison_cols].mean().T
comparison.columns = ['Renewed', 'Non-Renewed']
comparison['Difference (%)'] = ((comparison['Renewed'] - comparison['Non-Renewed']) / comparison['Non-Renewed'] * 100).round(1)
print(comparison)

# Visualize
fig, ax = plt.subplots(figsize=(12, 6))
comparison[['Renewed', 'Non-Renewed']].plot(kind='bar', ax=ax, color=['steelblue', 'coral'])
ax.set_title('Engagement Metrics: Renewed vs Non-Renewed Clients')
ax.set_ylabel('Average Value')
ax.tick_params(axis='x', rotation=45)
plt.legend(title='Renewal Status')
plt.tight_layout()
plt.savefig('../powerbi/screenshots/retention_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Renewal Rate by Engagement Band

In [ ]:
engagement_analysis = features_df.groupby('engagement_band').agg(
    total_customers=('customer_id', 'count'),
    renewed=('churn', lambda x: (x == 'No').sum()),
    churned=('churn', lambda x: (x == 'Yes').sum()),
    renewal_rate=('churn', lambda x: (x == 'No').mean() * 100),
    avg_monthly_charges=('monthly_charges', 'mean'),
    avg_tenure=('tenure', 'mean'),
).round(2)

print(engagement_analysis)

fig, ax = plt.subplots(figsize=(10, 6))
engagement_analysis['renewal_rate'].plot(kind='bar', color='teal', ax=ax)
ax.set_title('Renewal Rate by Engagement Band')
ax.set_ylabel('Renewal Rate (%)')
ax.set_xlabel('Engagement Band')
ax.tick_params(axis='x', rotation=0)
for i, v in enumerate(engagement_analysis['renewal_rate']):
    ax.text(i, v + 1, f'{v:.1f}%', ha='center', va='bottom')
plt.tight_layout()
plt.savefig('../powerbi/screenshots/renewal_by_engagement_band.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. At-Risk Client Identification

In [ ]:
# Define at-risk criteria
features_df['risk_interaction_low'] = features_df['total_interactions'] < features_df['total_interactions'].quantile(0.25)
features_df['risk_activity_low'] = features_df['avg_logins_per_day'] < features_df['avg_logins_per_day'].quantile(0.25)
features_df['risk_engagement_low'] = features_df['engagement_score'] < features_df['engagement_score'].quantile(0.25)

features_df['at_risk'] = (
    features_df['risk_interaction_low'] &
    features_df['risk_activity_low']
)

at_risk_count = features_df['at_risk'].sum()
print(f"At-risk clients: {at_risk_count} ({at_risk_count / len(features_df) * 100:.1f}%)")

at_risk_df = features_df[features_df['at_risk']].copy()
print(f"\nAt-risk client profile:")
print(f"  - Average engagement score: {at_risk_df['engagement_score'].mean():.3f}")
print(f"  - Average monthly charges: ${at_risk_df['monthly_charges'].mean():.2f}")
print(f"  - Average tenure: {at_risk_df['tenure'].mean():.1f} months")
print(f"  - Already churned: {(at_risk_df['churn'] == 'Yes').mean() * 100:.1f}%")

# Show top 10 at-risk clients by contract value
at_risk_display = at_risk_df[[
    'customer_id', 'contract', 'monthly_charges', 'engagement_score',
    'total_interactions', 'total_logins', 'churn'
]].sort_values('monthly_charges', ascending=False).head(10)
print("\nTop 10 At-Risk Clients (by monthly charges):")
print(at_risk_display)

## 5. Engagement Score Sensitivity

In [ ]:
# Compare engagement score versions
features_a = pd.read_csv(os.path.join(PROCESSED_DIR, "customer_features_vA.csv"))
features_c = pd.read_csv(os.path.join(PROCESSED_DIR, "customer_features_vC.csv"))

sensitivity = features_df[['customer_id', 'churn', 'engagement_score']].copy()
sensitivity = sensitivity.merge(
    features_a[['customer_id', 'engagement_score']].rename(columns={'engagement_score': 'score_vA'}),
    on='customer_id'
)
sensitivity = sensitivity.merge(
    features_c[['customer_id', 'engagement_score']].rename(columns={'engagement_score': 'score_vC'}),
    on='customer_id'
)

sensitivity.rename(columns={'engagement_score': 'score_vB'}, inplace=True)

print("Engagement Score Sensitivity Analysis:")
print(f"Correlation vA vs vB: {sensitivity['score_vA'].corr(sensitivity['score_vB']):.3f}")
print(f"Correlation vA vs vC: {sensitivity['score_vA'].corr(sensitivity['score_vC']):.3f}")
print(f"Correlation vB vs vC: {sensitivity['score_vB'].corr(sensitivity['score_vC']):.3f}")

# Check if conclusions change
for version in ['score_vA', 'score_vB', 'score_vC']:
    renewed_avg = sensitivity[sensitivity['churn'] == 'No'][version].mean()
    churned_avg = sensitivity[sensitivity['churn'] == 'Yes'][version].mean()
    print(f"\n{version}: Renewed avg = {renewed_avg:.3f}, Churned avg = {churned_avg:.3f}, Gap = {(renewed_avg - churned_avg):.3f}")